# Exercise 4 — Approximating MPC with a neural network

The MPC of Exercise 2 solves an optimisation problem at every time step. Here we **learn its control law** with a neural network:

$$u_k \approx \pi_{NN}(T_k,\ T_{o,k})$$

1. **Offline:** run the MPC in closed-loop simulations and record the states $(T_k, T_{o,k})$ and the MPC inputs $u_k$.
2. **Offline:** train a neural network on these data.
3. **Online:** replace the MPC with the network. One forward pass instead of one optimisation per step.

Same building, outdoor temperature (measured $T_o$ − 12 °C), comfort band and heater limit as before.

In [ ]:
# Google Colab only: download the course files and check the packages.
# When you run the notebook on your own computer, this cell does nothing.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("building_model.py"):
        if not os.path.exists("athens_neural_network_for_control"):
            subprocess.run(["git", "clone", "-q", "https://github.com/li-yun/athens_neural_network_for_control.git"], check=True)
        os.chdir("athens_neural_network_for_control")
    r = subprocess.run([sys.executable, "install_packages.py"], capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(out[out.find("2/3"):] if r.returncode == 0 else out)   # show the check results
    if r.returncode != 0:
        raise RuntimeError("Setup failed, see the messages above.")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from building_model import FirstOrderBuilding, outdoor_temperature, plot_closed_loop, evaluate

model = FirstOrderBuilding()
r, u_max = 21.0, 2000.0
T_min, T_max = 19.0, 24.0
N, lam = 48, 1e-6                                          # MPC settings of Exercise 2

To = outdoor_temperature('winter')
To_forecast = np.concatenate([To, np.full(N, To[-1])])

## The MPC (from Exercise 2, given)

In [ ]:
class MPC:
    """MPC for the 1R1C building, solved with cvxpy. Call: controller(r_k, T_k, k) -> u_k."""

    def __init__(self, model, N, lam, To_forecast, u_max=2000.0, T_min=19.0, T_max=24.0):
        self.N, self.To_forecast = N, To_forecast
        self.solve_times = []
        T = cp.Variable(N + 1)
        self.u = u = cp.Variable(N)
        self.T0, self.d, self.r = cp.Parameter(), cp.Parameter(N), cp.Parameter(N)
        J = cp.sum_squares(self.r - T[1:]) + lam * cp.sum_squares(u)
        constraints = [T[0] == self.T0,
                       T[1:] == model.A * T[:-1] + model.B * u + model.E * self.d,
                       u >= 0, u <= u_max, T[1:] >= T_min, T[1:] <= T_max]
        self.prob = cp.Problem(cp.Minimize(J), constraints)

    def __call__(self, r_k, T_k, k):
        self.T0.value = T_k
        self.d.value = self.To_forecast[k:k + self.N]
        self.r.value = np.full(self.N, r_k)
        t0 = time.perf_counter()
        self.prob.solve(solver=cp.CLARABEL)
        self.solve_times.append(time.perf_counter() - t0)
        return float(np.clip(self.u.value[0], 0.0, None))

mpc = MPC(model, N, lam, To_forecast)

## 1. Task: collect training data with the MPC (offline)

Run the MPC in closed loop from several initial temperatures. From every result `res` (a dict with arrays `res['T']`, `res['To']`, `res['u']`), store the network inputs $[T, T_o]$ in `X_list` and the MPC input $u$ in `y_list`.

In [ ]:
T0_train = [19.0, 19.5, 20.5, 21.0, 22.0, 23.0, 24.0]     # T0 = 20 is kept for testing
X_list, y_list = [], []

t0 = time.perf_counter()
for T0 in T0_train:
    res = model.simulate_closed_loop(T0, r, To, controller=mpc, pass_k=True)
    # TODO: append the inputs [T, To] (hint: np.column_stack) and the MPC input u (2 lines)
    ...

X_train, y_train = np.vstack(X_list), np.concatenate(y_list)
print(f'{len(y_train)} samples, generated in {time.perf_counter() - t0:.1f} s')

## 2. Task: train the neural network (offline)

Same structure as in Exercise 3: `StandardScaler` + `MLPRegressor` with one hidden layer of 16 neurons, `activation='tanh'`, `max_iter=5000, tol=1e-7, n_iter_no_change=50, random_state=0`.

Train on the heating power in **kW** (`y_train / 1000`): targets of order 1 train better than targets of order 1000.

In [ ]:
# TODO: create the network and fit it on X_train and y_train / 1000 (2 lines)
nn = ...

print(f'training RMSE: {1000 * np.sqrt(np.mean((nn.predict(X_train) - y_train / 1000)**2)):.0f} W')

## 3. Task: the neural-network controller (online)

Write the controller that replaces the MPC: predict $u$ from $[T_k, T_{o,k}]$, convert kW → W and limit it to $[0, u_{\max}]$.

*Hint:* `nn.predict([[T_k, To[k]]])` returns an array with one value.

In [ ]:
def nn_controller(r_k, T_k, k):
    # TODO: 2 lines
    return ...

## 4. The learned control law

MPC data points (coloured by outdoor temperature) and the network output for three outdoor temperatures. Where there are no MPC data (e.g. $T < 20.5$ °C at mild outdoor temperatures) the network only extrapolates, and its output there is not reliable.

In [ ]:
T_grid = np.linspace(18.5, 24.5, 100)
plt.figure(figsize=(8, 4))
sc = plt.scatter(X_train[:, 0], y_train, c=X_train[:, 1], s=8, cmap='coolwarm', label='MPC data')
for To_val, col in zip([3, 10, 17], ['tab:blue', 'tab:gray', 'tab:red']):
    X_grid = np.column_stack([T_grid, np.full_like(T_grid, To_val)])
    u_nn = np.clip(1000 * nn.predict(X_grid), 0, u_max)
    plt.plot(T_grid, u_nn, color=col, lw=2, label=f'NN, To = {To_val} °C')
plt.colorbar(sc, label='To [°C]')
plt.xlabel('indoor temperature T [°C]'); plt.ylabel('heating power u [W]')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. Online: MPC vs. neural network

Test from $T_0 = 20$ °C, which was not used for training.

In [ ]:
def timed(controller, times):
    def wrapped(r_k, T_k, k):
        t0 = time.perf_counter(); u = controller(r_k, T_k, k); times.append(time.perf_counter() - t0)
        return u
    return wrapped

T0_test = 20.0
t_mpc, t_nn = [], []
res_mpc = model.simulate_closed_loop(T0_test, r, To, controller=timed(MPC(model, N, lam, To_forecast), t_mpc), pass_k=True)
res_nn = model.simulate_closed_loop(T0_test, r, To, controller=timed(nn_controller, t_nn), pass_k=True)

plot_closed_loop([res_mpc, res_nn], labels=['MPC (online optimisation)', 'NN approximation'],
                 u_max=u_max, T_band=(T_min, T_max))
plt.show()

### Computation time

`nn.predict` carries a fixed overhead of sklearn (input checks, etc.) on every call. A deployed controller only evaluates the network itself: $u = W_2 \tanh(W_1 x_{scaled} + b_1) + b_2$. That is a few multiplications with the trained weights, written below in plain numpy.

In [ ]:
scaler, mlp = nn[0], nn[-1]
W1, W2 = mlp.coefs_
b1, b2 = mlp.intercepts_

def nn_forward(T_k, To_k):
    """Forward pass of the trained network with plain numpy (what runs on the controller)."""
    x = (np.array([T_k, To_k]) - scaler.mean_) / scaler.scale_
    return float(np.clip(1000 * (np.tanh(x @ W1 + b1) @ W2 + b2)[0], 0.0, u_max))

# same result as nn.predict
assert abs(nn_forward(20.5, 5.0) - np.clip(1000 * nn.predict([[20.5, 5.0]])[0], 0, u_max)) < 1e-6

t_fwd = []
res_fwd = model.simulate_closed_loop(T0_test, r, To, controller=timed(lambda r_k, T_k, k: nn_forward(T_k, To[k]), t_fwd), pass_k=True)

table = pd.DataFrame({
    'MPC (cvxpy)':           {**evaluate(res_mpc, T_min=T_min, T_max=T_max), 'time per step [ms]': 1e3 * np.mean(t_mpc[1:])},
    'NN (sklearn predict)':  {**evaluate(res_nn, T_min=T_min, T_max=T_max), 'time per step [ms]': 1e3 * np.mean(t_nn)},
    'NN (numpy forward)':    {**evaluate(res_fwd, T_min=T_min, T_max=T_max), 'time per step [ms]': 1e3 * np.mean(t_fwd)},
}).T
table['speed-up vs MPC'] = table.loc['MPC (cvxpy)', 'time per step [ms]'] / table['time per step [ms]']
table.round(4)

In [ ]:
plt.figure(figsize=(6, 3))
plt.barh(table.index, table['time per step [ms]'], color=['tab:blue', 'tab:orange', 'tab:green'])
plt.xscale('log'); plt.xlabel('computation time per step [ms] (log scale)'); plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()